### **PHASE 2 — Inspect and Parse Legal Document Structure**

Input: cleaned `.txt` files and `corpus_registry.json`

Output: structural inspection results and Article-level records

Purpose: Legal documents are parsed into stable structured units so QA evidence and retrieval results can be evaluated independently of the five chunking strategies.


In [2]:
import json, re
import pandas as pd
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")  # suppress all warnings

In [3]:
PROJECT_ROOT = Path("/Users/tanggiee/Desktop/RAG_AI/esg_rag_project")
REGISTRY_PATH = PROJECT_ROOT / "outputs" / "logs" / "corpus_registry.json"

registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
registry_df = pd.DataFrame(registry)

print("Registry documents:", len(registry_df))
print("Available cleaned files:", registry_df["cleaned_path"].apply(lambda path: Path(path).exists()).sum())

Registry documents: 403
Available cleaned files: 403


In [9]:
SAMPLE_TYPES = ["Law", "Decree", "Circular", "Decision", "Resolution", "Integrated Document"]
SAMPLES_PER_TYPE = 4
RANDOM_SEED = 43

OLD_MANIFEST_PATH = PROJECT_ROOT / "outputs" / "article_inspection_samples" / "sample_manifest.csv"
old_sample_filenames = set(pd.read_csv(OLD_MANIFEST_PATH)["source_filename"]) if OLD_MANIFEST_PATH.exists() else set()

candidates = registry_df[
    registry_df["document_type"].isin(SAMPLE_TYPES)
    & ~registry_df["source_filename"].isin(old_sample_filenames)].copy()

selected_indexes = []
for _, group in candidates.groupby("document_type"):
    selected = group.sample(min(SAMPLES_PER_TYPE, len(group)), random_state=RANDOM_SEED)
    selected_indexes.extend(selected.index)

additional_sample_df = candidates.loc[selected_indexes].reset_index(drop=True)

display(additional_sample_df[
    ["official_number", "document_type", "title", "word_count", "source_filename", "cleaned_path"]])

print("Previously inspected:", len(old_sample_filenames))
print("Additional documents:", len(additional_sample_df))

,official_number,document_type,title,word_count,source_filename,cleaned_path
0,04/2024/TT-BTNMT,Circular,ON AUDIT OF COMPLIANCE WITH LAWS ON WATER RESO...,6545,04_2024_TT-BTNMT_m_611481.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
1,08/2025/TT-BNNMT,Circular,PROVIDING AMENDMENTS TO CIRCULAR NO. 01/2022/T...,3326,08_2025_TT-BNNMT_m_666356.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
2,02/2025/TT-BTNMT,Circular,NATIONAL TECHNICAL REGULATION ON ENVIRONMENTAL...,3531,02_2025_TT-BTNMT_m_655384.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
3,38/2025/TT-BNNMT,Circular,ON METHODS FOR DETERMINATION OF REIMBURSED COS...,3481,38_2025_TT-BNNMT_m_674733.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
4,04/2017/QD-TTg,Decision,ON THE LIST OF EQUIPMENT AND APPLIANCES TO WHI...,1134,04_2017_QD-TTg_m_346090.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
5,882/QD-TTg,Decision,APPROVING THE NATIONAL ACTION PLAN FOR GREEN G...,59552,882_QD-TTg_m_579473.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
6,1690/QD-TTg,Decision,APPROVING VIETNAM'S FISHERIES DEVELOPMENT STRA...,4583,1690_QD-TTg_m_114738.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
7,19/2024/QD-TTg,Decision,ON THE ROADMAP FOR IMPLEMENTING EMISSION STAND...,1558,19_2024_QD-TTg_m_637377.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
8,243/2026/ND-CP,Decree,AMENDMENTS TO DECREE NO. 57/2025/ND-CP DATED M...,16707,243_2026_ND-CP_m_713627.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
9,37/2026/ND-CP,Decree,ELABORATING CERTAIN ARTICLES AND MEASURES FOR ...,39054,37_2026_ND-CP_695602.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...


Previously inspected: 12
Additional documents: 24


In [12]:
# Terms indicating document structures that commonly challenge legal parsers
DIFFICULT_TITLE_PATTERN = (
    r"\bAMEND(?:MENT|MENTS|ING)?\b|"
    r"\bSUPPLEMENT(?:ARY|ING)?\b|"
    r"\bREPLAC(?:E|ES|ED|EMENT|ING)\b|"
    r"\bCONSOLIDAT(?:E|ED|ING|ION)\b|"
    r"\bAPPENDIX\b|\bANNEX\b|"
    r"\bTECHNICAL REGULATION\b|"
    r"\bIMPLEMENTATION\b"
)

# Search only the remaining eligible documents, not the complete registry
title_edge_mask = candidates["title"].fillna("").str.contains(DIFFICULT_TITLE_PATTERN, case=False, regex=True)

# Integrated documents are difficult even when their titles are empty
integrated_mask = candidates["document_type"].eq("Integrated Document")

# Exclude documents already selected by the type-based random sample
already_sampled_mask = candidates["source_filename"].isin(additional_sample_df["source_filename"])

difficult_candidates = candidates[(title_edge_mask | integrated_mask) & ~already_sampled_mask].copy()

# Select up to eight difficult documents using a fixed reproducible seed
difficult_sample_df = difficult_candidates.sample(n=min(8, len(difficult_candidates)),random_state=44).reset_index(drop=True)

display(difficult_sample_df[
    ["official_number", "document_type", "title", "word_count", "source_filename", "cleaned_path"]])

print("Eligible difficult documents:", len(difficult_candidates))
print("Difficult documents selected:", len(difficult_sample_df))

,official_number,document_type,title,word_count,source_filename,cleaned_path
0,07/2025/TT-BTNMT,Circular,AMENDMENTS TO SOME ARTICLES OF CIRCULAR NO. 02...,12135,07_2025_TT-BTNMT_m_647200.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
1,08/2020/QD-TTg,Decision,AMENDMENTS TO DECISION NO. 24/2014/QD-TTG DATE...,1405,08_2020_QD-TTg_m_444837.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
2,09/2025/TT-BYT,Circular,ON NATIONAL TECHNICAL REGULATION ON PERMISSIBL...,8385,09_2025_TT-BYT_m_658051.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
3,01/2021/TT-BXD,Circular,"QCVN 01:2021/BXD, NATIONAL TECHNICAL REGULATIO...",21348,01_2021_TT-BXD_m_488637.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
4,73/2025/ND-CP,Decree,AMENDING PREFERENTIAL IMPORT TARIFF RATES ON S...,14843,73_2025_ND-CP_m_651068 (1).docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
5,69/2026/ND-CP,Decree,AMENDMENTS TO SOME ARTICLES OF THE GOVERNMENT’...,5437,69_2026_ND-CP_m_699320.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
6,79/2023/ND-CP,Decree,ELABORATING ON SEVERAL ARTICLES AND IMPLEMENTA...,11114,79_2023_ND-CP_m_590378.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
7,199/2025/ND-CP,Decree,ON AMENDMENTS TO DECREE NO. 26/2023/ND-CP DATE...,1800,199_2025_ND-CP_m_664759.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...


Eligible difficult documents: 80
Difficult documents selected: 8


In [13]:
# Collect filenames selected by the first two sampling methods
selected_filenames = (set(additional_sample_df["source_filename"]) | set(difficult_sample_df["source_filename"]))

# Keep only documents that have not already been selected
remaining_candidates = candidates[~candidates["source_filename"].isin(selected_filenames)].copy()

# Ensure word_count is numeric before comparing document lengths
remaining_candidates["word_count"] = pd.to_numeric(remaining_candidates["word_count"], errors="coerce")

# Exclude records with missing word counts
remaining_candidates = remaining_candidates.dropna(subset=["word_count"])

# Select four shortest and four longest remaining documents
short_sample_df = remaining_candidates.nsmallest(4, "word_count")
long_sample_df = remaining_candidates.nlargest(4, "word_count")

# Combine the two length-extreme groups
length_sample_df = pd.concat(
    [short_sample_df, long_sample_df],
    ignore_index=True).drop_duplicates("source_filename")

display(length_sample_df[
    ["official_number", "document_type", "title", "word_count", "source_filename", "cleaned_path"]])

print("Short documents selected:", len(short_sample_df))
print("Long documents selected:", len(long_sample_df))
print("Length-extreme documents:", len(length_sample_df))

,official_number,document_type,title,word_count,source_filename,cleaned_path
0,57/QD-TTg,Decision,APPROVING THE PLACEMENT OF CULTURAL ATTACHES I...,304,57_QD-TTg_m_102712.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
1,1009/QD-BCT,Decision,APPROVING THE GENERATION PRICE BRACKET FOR COA...,432,1009_QD-BCT_m_656831.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
2,144/2024/ND-CP,Decree,AMENDING THE GOVERNMENT’S DECREE NO. 26/2023/N...,442,144_2024_ND-CP_m_647300.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
3,21/QD-BCT,Decision,PROMULGATION OF THE TRANSITIONAL FRAMEWORK FOR...,471,21_QD-BCT_m_550028.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
4,768/QD-TTg,Decision,APPROVING AMENDMENT TO NATIONAL ELECTRICITY DE...,91305,768_QD-TTg_658055.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
5,08/2022/ND-CP,Decree,ELABORATION OF SEVERAL ARTICLES OF THE LAW ON ...,81535,08_2022_ND-CP_m_507203.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
6,356/2025/ND-CP,Decree,ELABORATING ON CERTAIN ARTICLES AND IMPLEMENTA...,63918,356_2025_ND-CP_689146.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...
7,262/QD-TTg,Decision,APPROVING THE PLAN TO IMPLEMENT THE NATIONAL P...,58534,262_QD-TTg_m_607766.docx,/Users/tanggiee/Desktop/RAG_AI/esg_rag_project...


Short documents selected: 4
Long documents selected: 4
Length-extreme documents: 8


In [14]:
# Combine the three complementary inspection samples:
# 1. random documents from each legal type;
# 2. deliberately difficult documents;
# 3. unusually short and long documents
expanded_sample_df = pd.concat([additional_sample_df, difficult_sample_df, length_sample_df],ignore_index=True)

# Remove any repeated filename that may appear in multiple categories
expanded_sample_df = (expanded_sample_df.drop_duplicates("source_filename").reset_index(drop=True))

# Record why each document was selected
type_sample_files = set(additional_sample_df["source_filename"])
difficult_sample_files = set(difficult_sample_df["source_filename"])
length_sample_files = set(length_sample_df["source_filename"])

def get_selection_reasons(filename):
    reasons = []
    if filename in type_sample_files:
        reasons.append("document_type_sample")
    if filename in difficult_sample_files:
        reasons.append("difficult_structure")
    if filename in length_sample_files:
        reasons.append("length_extreme")
    return reasons

expanded_sample_df["selection_reasons"] = expanded_sample_df[
    "source_filename"
].apply(get_selection_reasons)

display(expanded_sample_df[
    [
        "official_number",
        "document_type",
        "title",
        "word_count",
        "source_filename",
        "selection_reasons"
    ]
])

print("Type-based sample:", len(additional_sample_df))
print("Difficult sample:", len(difficult_sample_df))
print("Length-extreme sample:", len(length_sample_df))
print("Unique new documents:", len(expanded_sample_df))

,official_number,document_type,title,word_count,source_filename,selection_reasons
0,04/2024/TT-BTNMT,Circular,ON AUDIT OF COMPLIANCE WITH LAWS ON WATER RESO...,6545,04_2024_TT-BTNMT_m_611481.docx,[document_type_sample]
1,08/2025/TT-BNNMT,Circular,PROVIDING AMENDMENTS TO CIRCULAR NO. 01/2022/T...,3326,08_2025_TT-BNNMT_m_666356.docx,[document_type_sample]
2,02/2025/TT-BTNMT,Circular,NATIONAL TECHNICAL REGULATION ON ENVIRONMENTAL...,3531,02_2025_TT-BTNMT_m_655384.docx,[document_type_sample]
3,38/2025/TT-BNNMT,Circular,ON METHODS FOR DETERMINATION OF REIMBURSED COS...,3481,38_2025_TT-BNNMT_m_674733.docx,[document_type_sample]
4,04/2017/QD-TTg,Decision,ON THE LIST OF EQUIPMENT AND APPLIANCES TO WHI...,1134,04_2017_QD-TTg_m_346090.docx,[document_type_sample]
5,882/QD-TTg,Decision,APPROVING THE NATIONAL ACTION PLAN FOR GREEN G...,59552,882_QD-TTg_m_579473.docx,[document_type_sample]
6,1690/QD-TTg,Decision,APPROVING VIETNAM'S FISHERIES DEVELOPMENT STRA...,4583,1690_QD-TTg_m_114738.docx,[document_type_sample]
7,19/2024/QD-TTg,Decision,ON THE ROADMAP FOR IMPLEMENTING EMISSION STAND...,1558,19_2024_QD-TTg_m_637377.docx,[document_type_sample]
8,243/2026/ND-CP,Decree,AMENDMENTS TO DECREE NO. 57/2025/ND-CP DATED M...,16707,243_2026_ND-CP_m_713627.docx,[document_type_sample]
9,37/2026/ND-CP,Decree,ELABORATING CERTAIN ARTICLES AND MEASURES FOR ...,39054,37_2026_ND-CP_695602.docx,[document_type_sample]


Type-based sample: 24
Difficult sample: 8
Length-extreme sample: 8
Unique new documents: 40


In [15]:
# Check whether the expanded sample covers all important document types
document_type_summary = (
    expanded_sample_df["document_type"]
    .value_counts()
    .rename("document_count")
    .to_frame()
)

# Count how many documents were selected for each reason
selection_reason_summary = (
    expanded_sample_df["selection_reasons"]
    .explode()
    .value_counts()
    .rename("document_count")
    .to_frame()
)

display(document_type_summary)
display(selection_reason_summary)

print("Previously inspected documents:", len(old_sample_filenames))
print("New inspection documents:", len(expanded_sample_df))
print("Total inspected after expansion:",
      len(old_sample_filenames) + len(expanded_sample_df))

,document_count
document_type,
Decree,11
Decision,10
Circular,7
Integrated Document,4
Law,4
Resolution,4


,document_count
selection_reasons,
document_type_sample,24
difficult_structure,8
length_extreme,8


Previously inspected documents: 12
New inspection documents: 40
Total inspected after expansion: 52


In [16]:
import shutil

# Folder used only for the expanded inspection sample
EXPANDED_SAMPLE_FOLDER = (
    PROJECT_ROOT / "outputs" / "article_inspection_expanded"
)

EXPANDED_SAMPLE_FOLDER.mkdir(parents=True, exist_ok=True)

# Remove files from a previous run so the folder contains only the current sample
for old_file in EXPANDED_SAMPLE_FOLDER.iterdir():
    if old_file.is_file():
        old_file.unlink()

copied_files = []
missing_files = []

# Copy every selected cleaned text file into the inspection folder
for cleaned_path in expanded_sample_df["cleaned_path"]:
    source = Path(cleaned_path)

    if source.exists():
        shutil.copy2(source, EXPANDED_SAMPLE_FOLDER / source.name)
        copied_files.append(source.name)
    else:
        missing_files.append(str(source))

# Save a manifest describing each document and why it was selected
manifest_columns = [
    "official_number",
    "document_type",
    "title",
    "word_count",
    "source_filename",
    "cleaned_path",
    "selection_reasons"
]

expanded_sample_df[manifest_columns].to_csv(
    EXPANDED_SAMPLE_FOLDER / "expanded_sample_manifest.csv",
    index=False
)

# Create a ZIP file beside the sample folder
archive_path = shutil.make_archive(
    str(EXPANDED_SAMPLE_FOLDER),
    "zip",
    EXPANDED_SAMPLE_FOLDER
)

print("Documents selected:", len(expanded_sample_df))
print("Text files copied:", len(copied_files))
print("Missing text files:", len(missing_files))
print("ZIP created:", archive_path)

if missing_files:
    print("\nMissing files:")
    for missing_file in missing_files:
        print(missing_file)

Documents selected: 40
Text files copied: 40
Missing text files: 0
ZIP created: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/article_inspection_expanded.zip


#### **Inspection**

In [22]:
# Use the expanded sample when available; otherwise inspect the original 12-document sample
EXPANDED_FOLDER = PROJECT_ROOT / "outputs" / "article_inspection_expanded"
ORIGINAL_FOLDER = PROJECT_ROOT / "outputs" / "article_inspection_samples"
INSPECTION_FOLDER = EXPANDED_FOLDER if EXPANDED_FOLDER.exists() and list(EXPANDED_FOLDER.glob("*.txt")) else ORIGINAL_FOLDER

# Detect legal structures without yet deciding the final Article boundaries
# Detect candidate legal structures; final Article boundaries will be decided by the parser
STRUCTURE_PATTERNS = {
    "quoted_article": re.compile(r'^\s*["“‘]\s*Article\s+(\d+[A-Za-z]*)\s*[.:\-–—]?\s*(.*)$', re.I),
    "article": re.compile(r"^\s*Article\s+(\d+[A-Za-z]*)\s*[.:\-–—]?\s*(.*)$", re.I),
    "chapter": re.compile(r"^\s*Chapter\s+([IVXLCDM]+|\d+)\s*[.:\-–—]?\s*(.*)$", re.I),
    "section": re.compile(r"^\s*Section\s+([IVXLCDM]+|\d+)\s*[.:\-–—]?\s*(.*)$", re.I),
    "part": re.compile(r"^\s*Part\s+([IVXLCDM]+|\d+)\s*[.:\-–—]?\s*(.*)$", re.I),
    "roman_heading": re.compile(r"^\s*([IVXLCDM]+)\s*[.:\-–—]\s*([A-Z][A-Z0-9 ,/&()'’\-–—]+)$"),
    "numbered_clause": re.compile(r"^\s*(\d+)\.\s+(.+)$"),
    "lettered_point": re.compile(r"^\s*([a-z]{1,2})[)/.]\s+(.+)$", re.I),
    "appendix": re.compile(r"^\s*(APPENDIX|ANNEX)\b\s*(.*)$", re.I),
    "schedule": re.compile(r"^\s*SCHEDULE\s+([A-Z0-9.-]+)?\s*[.:\-–—]?\s*(.*)$", re.I),
    "footnote": re.compile(r"^\s*\[(\d+)\]\s*(.+)$"),
    "signature": re.compile(r"^\s*TABLE ROW\s*\|\s*\|.*\b(PP\.|ON BEHALF OF|FOR THE|DEPUTY PRIME MINISTER|MINISTER|PRESIDENT|CHAIRMAN|CHAIRPERSON|CERTIFIED BY|SIGNATURE|FULL NAME|SEAL)\b.*$", re.I)
}

print("Inspection folder:", INSPECTION_FOLDER)
print("Text files:", len(list(INSPECTION_FOLDER.glob("*.txt"))))

Inspection folder: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/article_inspection_expanded
Text files: 40


In [23]:
# Inspect every line, record its character position and preserve nearby context
def inspect_structure(path: Path, context_lines: int = 1) -> list:
    text = path.read_text(encoding="utf-8", errors="replace")
    lines = text.splitlines(keepends=True)
    offsets, position = [], 0 #calculate where every line starts 
    for line in lines:
        offsets.append(position)
        position += len(line)

    matches = []
    for index, raw_line in enumerate(lines):
        line = " ".join(raw_line.split())
        if not line:
            continue
        for structure_type, pattern in STRUCTURE_PATTERNS.items():
            match = pattern.match(line)
            if not match:
                continue

            # capture nearby context (by 1 line)
            start, end = max(0, index - context_lines), min(len(lines), index + context_lines + 1)
            context = " | ".join(" ".join(value.split()) for value in lines[start:end] if value.strip())
            matches.append({"source_filename": path.name, "structure_type": structure_type, "line_number": index + 1, "start_char": offsets[index], "matched_text": line, "context": context})
            break
    return matches

# Run the inspection across all selected cleaned documents
inspection_records = []
for path in sorted(INSPECTION_FOLDER.glob("*.txt")):
    try:
        inspection_records.extend(inspect_structure(path))
    except Exception as error:
        print(f"ERROR | {path.name} | {error}")

inspection_df = pd.DataFrame(inspection_records)
print("Documents inspected:", len(list(INSPECTION_FOLDER.glob("*.txt"))))
print("Structural matches:", len(inspection_df))
display(inspection_df.head(50))

Documents inspected: 40
Structural matches: 14020


,source_filename,structure_type,line_number,start_char,matched_text,context
0,01_2021_TT-BXD_m_488637.txt,article,10,932,Article 1. Attached to this Circular are the N...,Minister of Construction promulgates Circular ...
1,01_2021_TT-BXD_m_488637.txt,article,11,1058,Article 2. This Circular comes into effect fro...,Article 1. Attached to this Circular are the N...
2,01_2021_TT-BXD_m_488637.txt,article,12,1269,"Article 3. Ministries, ministerial agencies, G...",Article 2. This Circular comes into effect fro...
3,01_2021_TT-BXD_m_488637.txt,signature,13,1499,TABLE ROW | | PP. MINISTER DEPUTY MINISTERLe Q...,"Article 3. Ministries, ministerial agencies, G..."
4,01_2021_TT-BXD_m_488637.txt,numbered_clause,40,2672,3. Regulations on management,2.16 Requirements for construction planning in...
5,01_2021_TT-BXD_m_488637.txt,numbered_clause,41,2701,4. Responsibilities of organizations and indiv...,3. Regulations on management | 4. Responsibili...
6,01_2021_TT-BXD_m_488637.txt,numbered_clause,42,2754,5. Organization for implementation,4. Responsibilities of organizations and indiv...
7,01_2021_TT-BXD_m_488637.txt,numbered_clause,47,3332,1. GENERAL PROVISIONS,NATIONAL TECHNICAL REGULATION ON CONSTRUCTION ...
8,01_2021_TT-BXD_m_488637.txt,numbered_clause,207,24507,2. TECHNICAL REGULATIONS,- 1/500 map illustrating roads leading to resi...
9,01_2021_TT-BXD_m_488637.txt,schedule,210,24898,Schedule 2.1: Average land criteria of urban a...,Minimum and maximum requirements for civil lan...


In [19]:
# Count every structural type found in each document
structure_summary = pd.crosstab(inspection_df["source_filename"], inspection_df["structure_type"]).reset_index()

# Identify documents with no ordinary or quoted Article headings
article_files = set(inspection_df.loc[inspection_df["structure_type"].isin(["article", "quoted_article"]), "source_filename"])
all_files = {path.name for path in INSPECTION_FOLDER.glob("*.txt")}
documents_without_articles = pd.DataFrame({"source_filename": sorted(all_files - article_files)})

display(structure_summary)
display(documents_without_articles)

print("Documents without Articles:", len(documents_without_articles))

structure_type,source_filename,appendix,article,chapter,footnote,lettered_point,numbered_clause,quoted_article,roman_heading,section,signature
0,01_2021_TT-BXD_m_488637.txt,32,3,0,0,0,8,0,0,0,4
1,02_2025_TT-BTNMT_m_655384.txt,0,4,0,0,2,10,0,0,0,4
2,04_2017_QD-TTg_m_346090.txt,0,6,0,0,18,23,0,0,0,2
3,04_2024_TT-BTNMT_m_611481.txt,0,28,4,0,68,92,0,0,4,1
4,07_2017_QH14_m_355880.txt,0,60,6,0,186,264,0,0,4,1
5,07_2025_TT-BTNMT_m_647200.txt,1,5,0,0,68,104,11,5,0,17
6,08_2020_QD-TTg_m_444837.txt,0,2,0,0,7,33,5,0,0,4
7,08_2022_ND-CP_m_507203.txt,0,165,13,0,1096,805,1,0,35,15
8,08_2025_TT-BNNMT_m_666356.txt,0,5,0,0,74,47,9,0,0,2
9,09_2025_TT-BYT_m_658051.txt,1,4,0,0,2,11,0,0,0,1


,source_filename
0,1690_QD-TTg_m_114738.txt


Documents without Articles: 1


In [24]:
# Show up to ten examples of every detected structure for manual validation
for structure_type in STRUCTURE_PATTERNS:
    examples = inspection_df[inspection_df["structure_type"].eq(structure_type)]
    print(f"\n{structure_type.upper()}: {len(examples)}")
    display(examples[["source_filename", "line_number", "matched_text", "context"]].head(10))


QUOTED_ARTICLE: 72


,source_filename,line_number,matched_text,context
836,07_2025_TT-BTNMT_m_647200.txt,30,“Article 18. Working principles and responsibi...,5. Article 18 shall be amended as follows: | “...
858,07_2025_TT-BTNMT_m_647200.txt,52,“Article 19. Forms of documents used for issua...,6. Article 19 shall be amended as follows: | “...
885,07_2025_TT-BTNMT_m_647200.txt,79,“Article 20. Additional waste monitoring regar...,7. Article 20 shall be amended as follows: | “...
903,07_2025_TT-BTNMT_m_647200.txt,104,“Article 22. Environmental registration form,9. Article 22 shall be amended as follows: | “...
905,07_2025_TT-BTNMT_m_647200.txt,107,“Article 23. Environmental registration receipt,10. Article 23 shall be amended as follows: | ...
909,07_2025_TT-BTNMT_m_647200.txt,111,“Article 25a. Environmental self-auditing by b...,11. Article 25a shall be added after Article 2...
923,07_2025_TT-BTNMT_m_647200.txt,125,“Article 26a. Classification of other domestic...,12. Article 26a shall be added after Article 2...
932,07_2025_TT-BTNMT_m_647200.txt,138,"""Article 31. Method of valuation of domestic s...",15. Article 31 shall be amended as follows: | ...
934,07_2025_TT-BTNMT_m_647200.txt,141,“Article 39a. Environmental management plan pr...,16. Article 39a shall be added after Article 3...
957,07_2025_TT-BTNMT_m_647200.txt,168,“Article 78. Responsibility of producer/import...,20. Article 78 shall be amended as follows: | ...



ARTICLE: 1713


,source_filename,line_number,matched_text,context
0,01_2021_TT-BXD_m_488637.txt,10,Article 1. Attached to this Circular are the N...,Minister of Construction promulgates Circular ...
1,01_2021_TT-BXD_m_488637.txt,11,Article 2. This Circular comes into effect fro...,Article 1. Attached to this Circular are the N...
2,01_2021_TT-BXD_m_488637.txt,12,"Article 3. Ministries, ministerial agencies, G...",Article 2. This Circular comes into effect fro...
44,02_2025_TT-BTNMT_m_655384.txt,12,Article 1. The QCVN 01:2025/BTNMT National tec...,The Minister of Natural Resources and Environm...
45,02_2025_TT-BTNMT_m_655384.txt,13,Article 2. Entry into force,Article 1. The QCVN 01:2025/BTNMT National tec...
46,02_2025_TT-BTNMT_m_655384.txt,15,Article 3. Transitional provisions,This Circular comes into force 6 months after ...
51,02_2025_TT-BTNMT_m_655384.txt,20,Article 4. Implementation,b) Relocate the facilities to other places mee...
60,04_2017_QD-TTg_m_346090.txt,10,Article 1: The list of equipment and appliance...,The Prime Minister has promulgated the Decisio...
65,04_2017_QD-TTg_m_346090.txt,15,Article 2: Roadmap to energy labeling,4. The regulated means of transport include: a...
80,04_2017_QD-TTg_m_346090.txt,30,Article 3: Roadmap to application of the minim...,4. Energy labels for the equipment and applian...



CHAPTER: 140


...

,source_filename,line_number,matched_text,context
9,01_2021_TT-BXD_m_488637.txt,210,Schedule 2.1: Average land criteria of urban a...,Minimum and maximum requirements for civil lan...
10,01_2021_TT-BXD_m_488637.txt,219,Schedule 2.2: Average land criteria for reside...,- Average land for residence-related units in ...
11,01_2021_TT-BXD_m_488637.txt,238,Schedule 2.3: Minimum scale of urban level ser...,Urban level service - public structures must c...
12,01_2021_TT-BXD_m_488637.txt,260,Schedule 2.4: Minimum scale of service - publi...,- Public - service structures of residence-rel...
13,01_2021_TT-BXD_m_488637.txt,280,Schedule 2.5: Minimum area of tree planting la...,- Urban areas that have characteristic and val...
14,01_2021_TT-BXD_m_488637.txt,303,Schedule 2.6: Minimum percentage of land for t...,- Maximum net building density of land plot fo...
15,01_2021_TT-BXD_m_488637.txt,323,Schedule 2.7: Minimum clearance (m) of structu...,- With respect to a structure consisting of a ...
16,01_2021_TT-BXD_m_488637.txt,334,Schedule 2.8: Maximum net building density of ...,- Maximum net building density of land plots f...
17,01_2021_TT-BXD_m_488637.txt,338,Schedule 2.9: Maximum net building density of ...,TABLE ROW | NOTE: Land plot for construction o...
18,01_2021_TT-BXD_m_488637.txt,354,Schedule 2.10: Maximum net building density of...,TABLE ROW | NOTE: For land plots with structur...



FOOTNOTE: 5


,source_filename,line_number,matched_text,context
8039,19_VBHN-VPQH_m_462843.txt,410,[1] The Law No. 61/2014/QH13 dated November 21...,TABLE ROW | | AUTHENTICATION OF CONSOLIDATED D...
8040,19_VBHN-VPQH_m_462843.txt,416,[2] This point is amended by Article 2 of the ...,The National Assembly hereby promulgates the L...
8041,19_VBHN-VPQH_m_462843.txt,417,[3] This point is amended by Article 2 of the ...,[2] This point is amended by Article 2 of the ...
8042,19_VBHN-VPQH_m_462843.txt,418,[4] This clause is amended by clause 3 Article...,[3] This point is amended by Article 2 of the ...
8043,19_VBHN-VPQH_m_462843.txt,419,[5] Article 2 of the Law No. 61/2014/QH13 read...,[4] This clause is amended by clause 3 Article...



SIGNATURE: 55


,source_filename,line_number,matched_text,context
3,01_2021_TT-BXD_m_488637.txt,13,TABLE ROW | | PP. MINISTER DEPUTY MINISTERLe Q...,"Article 3. Ministries, ministerial agencies, G..."
823,07_2017_QH14_m_355880.txt,553,TABLE ROW | | CHAIRMAN OF THE NATIONAL ASSEMBL...,This Law has been ratified in the 3rd session ...
1001,07_2025_TT-BTNMT_m_647200.txt,229,"TABLE ROW | | Legal representative(Signature, ...",(Name of the producer/importer) registers a pl...
1002,07_2025_TT-BTNMT_m_647200.txt,265,"TABLE ROW | | Legal representative(Signature, ...",(Name of the producer/importer) reports result...
1004,07_2025_TT-BTNMT_m_647200.txt,302,"TABLE ROW | | Legal representative(Signature, ...",(Name of the authorized party) reports result ...
1009,07_2025_TT-BTNMT_m_647200.txt,351,"TABLE ROW | | Legal representative(Signature, ...",- Make a commitment that products/packaging re...
1017,07_2025_TT-BTNMT_m_647200.txt,412,"TABLE ROW | | Legal representative(Signature, ...",(The producer/importer(self-recycling)/the rec...
1018,07_2025_TT-BTNMT_m_647200.txt,435,"TABLE ROW | | Legal representative(Signature, ...",(Name of the producer/importer) commits to tak...
1019,07_2025_TT-BTNMT_m_647200.txt,463,"TABLE ROW | | Legal representative(Signature, ...",(Name of the producer/importer) commits to tak...
1020,07_2025_TT-BTNMT_m_647200.txt,488,"TABLE ROW | | Legal representative(Signature, ...",(Name of the producer/importer) commits to tak...


In [25]:
# Save detailed matches and summaries so the inspection does not need to be repeated
INSPECTION_OUTPUT_FOLDER = PROJECT_ROOT / "outputs" / "article_structure_inspection"
INSPECTION_OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

inspection_df.to_csv(INSPECTION_OUTPUT_FOLDER / "structure_matches.csv", index=False)
structure_summary.to_csv(INSPECTION_OUTPUT_FOLDER / "structure_summary.csv", index=False)
documents_without_articles.to_csv(INSPECTION_OUTPUT_FOLDER / "documents_without_articles.csv", index=False)

print("Inspection results saved:", INSPECTION_OUTPUT_FOLDER)

Inspection results saved: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/article_structure_inspection


### **2.1. Legal parser** 

#### Inspect legal structure 

In [26]:
# Reuse the existing inspection folder and create a separate folder for parser outputs
PARSER_REVIEW_FOLDER = PROJECT_ROOT / "outputs" / "article_parser_review"
PARSER_REVIEW_FOLDER.mkdir(parents=True, exist_ok=True)

# Stop early if the inspection input folder or its cleaned text files are unavailable
assert "INSPECTION_FOLDER" in globals(), "Run the inspection-folder configuration cell first."
inspection_files = sorted(INSPECTION_FOLDER.glob("*.txt"))
assert inspection_files, f"No cleaned .txt files found in {INSPECTION_FOLDER}"

print("Parser input folder:", INSPECTION_FOLDER)
print("Documents available:", len(inspection_files))
print("Parser output folder:", PARSER_REVIEW_FOLDER)

Parser input folder: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/article_inspection_expanded
Documents available: 40
Parser output folder: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/article_parser_review


In [81]:
# Detect the official hierarchy used in English-translated Vietnamese legal documents
PART_PATTERN = re.compile(r"^\s*Part\s+([IVXLCDM]+)\s*[.:\-–—]?\s*(.*)$", re.I)
CHAPTER_PATTERN = re.compile(r"^\s*Chapter\s+([IVXLCDM]+)\s*[.:\-–—]?\s*(.*)$", re.I)
SECTION_PATTERN = re.compile(r"^\s*Section\s+(\d+[A-Za-z]*)\s*[.:\-–—]?\s*(.*)$", re.I)
SUBSECTION_PATTERN = re.compile(r"^\s*Subsection\s+(\d+[A-Za-z]*)\s*[.:\-–—]?\s*(.*)$", re.I)
ARTICLE_PATTERN = re.compile(r"^\s*Article\s+(\d+[A-Za-z]*)\s*[.:\-–—]?\s*(.*)$", re.I)
QUOTED_ARTICLE_PATTERN = re.compile(r'^\s*["“‘]\s*Article\s+(\d+[A-Za-z]*)\s*[.:\-–—]?\s*(.*)$', re.I)
CLAUSE_PATTERN = re.compile(r"^\s*(\d+[A-Za-z]*)\.\s+(.+)$", re.I)
POINT_PATTERN = re.compile(r"^\s*([A-Za-z]{1,2})[)/]\s+(.+)$", re.I)
APPENDIX_PATTERN = re.compile(r"^\s*(?:Appendix|Annex)\s*([IVXLCDM]+|\d+)?\s*[.:\-–—]?\s*(.*)$", re.I)
FOOTNOTE_PATTERN = re.compile(r"^\s*\[(\d+)\]\s*(.+)$")
ROMAN_PATTERN = re.compile(r"^\s*([IVXLCDM]+)\s*[.:\-–—]\s*([A-Z][A-Z0-9 ,/&()'’\-–—]+)$")
SIGNATURE_PATTERN = re.compile(r"^\s*TABLE ROW\s*\|\s*\|.*\b(PP\.|ON BEHALF OF|FOR THE|DEPUTY PRIME MINISTER|MINISTER|PRESIDENT|CHAIRMAN|CHAIRPERSON|CERTIFIED BY|SIGNATURE|FULL NAME|SEAL)\b.*$", re.I)

In [82]:
# Detect an amended Article embedded after an outer Article reference on the same line
INLINE_EMBEDDED_ARTICLE_PATTERN = re.compile(r'^\s*Article\s+\d+[A-Za-z]*\s*[.:\-–—]?\s*[“"]\s*Article\s+(\d+[A-Za-z]*)\s*[.:\-–—]?\s*(.*)$', re.I)

# Titles commonly used by outer Articles in amendment documents
AMENDMENT_ACTION_PATTERN = re.compile(r"\b(amend|add|addition|annul|repeal|replace|transition|effect|implement|entry|commence)\w*\b", re.I)

# Convert ordinary Article numbers to integers; alphanumeric numbers such as 13a return None
def numeric_article_number(value):
    return int(value) if str(value).isdigit() else None

In [83]:
# Return the next non-empty line when a structural title is written separately
def get_next_nonempty_line(lines: list, index: int, limit: int = 3) -> str:
    for next_index in range(index + 1, min(index + limit + 1, len(lines))):
        candidate = " ".join(lines[next_index].split())
        if candidate:
            return candidate
    return ""

In [129]:
# Parse one document into stable Articles/Roman sections and complete child provisions
def parse_legal_document(path: Path, oversized_threshold: int = 5000) -> tuple:
    text = path.read_text(encoding="utf-8", errors="replace")
    raw_lines = text.splitlines(keepends=True)
    lines = [" ".join(line.split()) for line in raw_lines]

    # Record the original character position of every line
    offsets, position = [], 0
    for raw_line in raw_lines:
        offsets.append(position)
        position += len(raw_line)

    # Store structural markers and current legal hierarchy
    article_markers, embedded_markers, roman_markers, boundaries = [], [], [], []
    part_number = part_title = chapter_number = chapter_title = ""
    section_number = section_title = subsection_number = subsection_title = ""
    
    # Track quotations, document regions and the sequential outer-Article structure
    article_seen, appendix_region, footnote_region = False, False, False
    is_integrated = "VBHN" in path.stem.upper()
    last_outer_number, amendment_mode = None, False

    # Create a top-level Article marker with its current hierarchy
    def make_article_marker(match, index):
        return {"index": index, 
                "start_char": offsets[index], 
                "unit_number": match.group(1), 
                "unit_title": match.group(2).strip() 
                or get_next_nonempty_line(lines, index), 
                "part_number": part_number, 
                "part_title": part_title, 
                "chapter_number": chapter_number, 
                "chapter_title": chapter_title, 
                "section_number": section_number, 
                "section_title": section_title, 
                "subsection_number": subsection_number, 
                "subsection_title": subsection_title}
    
    # Record a structural boundary once at its original character position
    def add_boundary(index, boundary_type):
        if not boundaries or boundaries[-1]["index"] != index:
            boundaries.append({"index": index, "start_char": offsets[index], "boundary_type": boundary_type})

    # Find the closing quotation of an embedded Article without crossing its parent-unit boundary
    def find_embedded_quote_end(start_char, maximum_end):
        segment = text[start_char:maximum_end]
        openings = [(segment.find(mark), mark) for mark in ["“", "‘", '"'] if segment.find(mark) >= 0]
        if not openings:
            return None

        opening_position, opening_mark = min(openings)
        closing_mark = {"“": "”", "‘": "’", '"': '"'}[opening_mark]

        # Curly quotations may contain nested quoted terms
        if opening_mark != '"':
            depth = 0
            for position in range(opening_position, len(segment)):
                if segment[position] == opening_mark:
                    depth += 1
                elif segment[position] == closing_mark:
                    depth -= 1
                    if depth == 0:
                        end = position + 1
                        while end < len(segment) and segment[end] in ".,;:":
                            end += 1
                        return start_char + end
            return None

        # Straight quotations use the next straight quote as the closing boundary
        closing_position = segment.find('"', opening_position + 1)
        if closing_position < 0:
            return None
        end = closing_position + 1
        while end < len(segment) and segment[end] in ".,;:":
            end += 1
        return start_char + end

    for index, line in enumerate(lines):
        if not line:
            continue

        # Mark where appendix, footnote or signature material begins
        if APPENDIX_PATTERN.match(line):
            appendix_region = True
            boundaries.append({"index": index, "start_char": offsets[index]})
        if FOOTNOTE_PATTERN.match(line) and article_seen:
            footnote_region = True
            boundaries.append({"index": index, "start_char": offsets[index]})
        if SIGNATURE_PATTERN.match(line):
            boundaries.append({"index": index, "start_char": offsets[index]})

       # Update hierarchy and prevent the preceding Article from absorbing a new hierarchy heading
        part_match = PART_PATTERN.match(line)
        if part_match and not appendix_region and not footnote_region:
            if article_seen:
                add_boundary(index, "part")
            part_number, part_title = part_match.group(1), part_match.group(2).strip() or get_next_nonempty_line(lines, index)
            chapter_number = chapter_title = section_number = section_title = subsection_number = subsection_title = ""

        chapter_match = CHAPTER_PATTERN.match(line)
        if chapter_match and not appendix_region and not footnote_region:
            if article_seen:
                add_boundary(index, "chapter")
            chapter_number, chapter_title = chapter_match.group(1), chapter_match.group(2).strip() or get_next_nonempty_line(lines, index)
            section_number = section_title = subsection_number = subsection_title = ""

        section_match = SECTION_PATTERN.match(line)
        if section_match and not appendix_region and not footnote_region:
            if article_seen:
                add_boundary(index, "section")
            section_number, section_title = section_match.group(1), section_match.group(2).strip() or get_next_nonempty_line(lines, index)
            subsection_number = subsection_title = ""

        subsection_match = SUBSECTION_PATTERN.match(line)
        if subsection_match and not appendix_region and not footnote_region:
            if article_seen:
                add_boundary(index, "subsection")
            subsection_number, subsection_title = subsection_match.group(1), subsection_match.group(2).strip() or get_next_nonempty_line(lines, index)

        # Detect ordinary, quoted and same-line embedded Article headings
        inline_match = INLINE_EMBEDDED_ARTICLE_PATTERN.match(line)
        quoted_match = QUOTED_ARTICLE_PATTERN.match(line)
        article_match = ARTICLE_PATTERN.match(line)
        candidate_match = quoted_match or article_match
        candidate_number = numeric_article_number(candidate_match.group(1)) if candidate_match else None
        candidate_title = candidate_match.group(2).strip() if candidate_match else ""
        in_legal_body = not appendix_region and not footnote_region

        # Correct a probable mistranslated Section only when surrounding numbering confirms it
        next_article_match = ARTICLE_PATTERN.match(get_next_nonempty_line(lines, index))
        next_number = numeric_article_number(next_article_match.group(1)) if next_article_match else None
        pseudo_section = bool(not is_integrated and article_match and not inline_match and last_outer_number is not None and candidate_number is not None and candidate_number <= last_outer_number and candidate_title.isupper() and next_number == last_outer_number + 1)

        # In amendment documents, outer Articles continue sequentially and normally use amendment-action titles
        expected_number = last_outer_number + 1 if last_outer_number is not None else None
        sequential_outer = bool(amendment_mode and candidate_number == expected_number and AMENDMENT_ACTION_PATTERN.search(candidate_title))

        if inline_match and in_legal_body:
            # Preserve the inner Article as an embedded provision
            embedded_markers.append({"index": index, "start_char": offsets[index], "unit_number": inline_match.group(1), "unit_title": inline_match.group(2).strip() or get_next_nonempty_line(lines, index)})

        elif pseudo_section and in_legal_body:
            # Preserve the heading as a boundary and assign its hierarchy to following Articles
            add_boundary(index, "corrected_section")
            section_number, section_title = article_match.group(1), candidate_title
            subsection_number = subsection_title = ""

        elif candidate_match and amendment_mode and in_legal_body:
            if sequential_outer:
                article_seen = True
                article_markers.append(make_article_marker(candidate_match, index))
                last_outer_number = candidate_number
            else:
                embedded_markers.append({"index": index, "start_char": offsets[index], "unit_number": candidate_match.group(1), "unit_title": candidate_title or get_next_nonempty_line(lines, index)})

        elif quoted_match and in_legal_body:
            embedded_markers.append({"index": index, "start_char": offsets[index], "unit_number": quoted_match.group(1), "unit_title": candidate_title or get_next_nonempty_line(lines, index)})

        elif article_match and in_legal_body:
            article_seen = True
            article_markers.append(make_article_marker(article_match, index))
            last_outer_number = candidate_number
            if len(article_markers) == 1 and not is_integrated and AMENDMENT_ACTION_PATTERN.search(candidate_title):
                amendment_mode = True

        # Use Roman headings only as fallback when the document has no Articles
        roman_match = ROMAN_PATTERN.match(line)
        if roman_match and not appendix_region and not footnote_region:
            roman_markers.append({"index": index, "start_char": offsets[index], 
                                  "unit_number": roman_match.group(1), 
                                  "unit_title": roman_match.group(2).strip(), 
                                  "part_number": "", "part_title": "", 
                                  "chapter_number": "", "chapter_title": "", 
                                  "section_number": "", "section_title": "", 
                                  "subsection_number": "", "subsection_title": ""})

    # Prefer Articles, then Roman sections; otherwise preserve the complete document
    if article_markers:
        markers, unit_type = article_markers, "article"
    elif roman_markers:
        markers, unit_type = roman_markers, "roman_section"
    else:
        markers, unit_type = [{"index": 0, "start_char": 0, 
                               "unit_number": "", "unit_title": "Complete document", 
                               "part_number": "", "part_title": "", 
                               "chapter_number": "", "chapter_title": "", 
                               "section_number": "", "section_title": "", 
                               "subsection_number": "", "subsection_title": ""}], "document_fallback"

    # Count repeated legal numbers before creating units
    number_totals = {}
    for marker in markers:
        number_key = marker["unit_number"].lower()
        number_totals[number_key] = number_totals.get(number_key, 0) + 1

    units, provisions, number_occurrences = [], [], {}

    for unit_index, marker in enumerate(markers):
        # Preserve the complete text for fallback documents; otherwise use legal boundaries
        if unit_type == "document_fallback":
            end_index, end_char = len(lines), len(text)
        else:
            later_markers = markers[unit_index + 1:]
            later_boundaries = [boundary for boundary in boundaries if boundary["index"] > marker["index"]]
            end_index = min([item["index"] for item in later_markers + later_boundaries] + [len(lines)])
            end_char = min([item["start_char"] for item in later_markers + later_boundaries] + [len(text)])

        start_char = marker["start_char"]
        unit_id = f"{path.stem}_{unit_type}_{unit_index + 1:04}"
        unit_text = text[start_char:end_char].strip()
        number_key = marker["unit_number"].lower()
        number_occurrences[number_key] = number_occurrences.get(number_key, 0) + 1

        # Save the complete stable Article or Roman-section unit
        units.append({"unit_id": unit_id, "doc_id": path.stem, "source_filename": path.name, "unit_type": unit_type, "unit_number": marker["unit_number"], "unit_number_occurrence": number_occurrences[number_key], "has_repeated_number": number_totals[number_key] > 1, "unit_title": marker["unit_title"], "part_number": marker["part_number"], "part_title": marker["part_title"], "chapter_number": marker["chapter_number"], "chapter_title": marker["chapter_title"], "section_number": marker["section_number"], "section_title": marker["section_title"], "subsection_number": marker["subsection_number"], "subsection_title": marker["subsection_title"], "start_char": start_char, "end_char": end_char, "char_count": end_char - start_char, "is_oversized": end_char - start_char > oversized_threshold, "unit_text": unit_text})

        # Find embedded Articles, Clauses and Points inside this unit
        embedded_by_index = {item["index"]: item for item in embedded_markers if marker["index"] < item["index"] < end_index}
        provision_markers = []

        for line_index in range(marker["index"] + 1, end_index):
            line = lines[line_index]
            if not line:
                continue

            if line_index in embedded_by_index:
                embedded = embedded_by_index[line_index]
                provision_markers.append({"index": line_index, "start_char": offsets[line_index], "provision_type": "embedded_article", "provision_number": embedded["unit_number"], "provision_title": embedded["unit_title"]})
                continue

            clause_match = CLAUSE_PATTERN.match(line)
            if clause_match:
                provision_markers.append({"index": line_index, "start_char": offsets[line_index], "provision_type": "clause" if unit_type == "article" else "numbered_item", "provision_number": clause_match.group(1), "provision_title": ""})
                continue

            point_match = POINT_PATTERN.match(line)
            if point_match:
                provision_markers.append({"index": line_index, "start_char": offsets[line_index], "provision_type": "point" if unit_type == "article" else "lettered_item", "provision_number": point_match.group(1), "provision_title": ""})

        # Create stable provision IDs and complete provision boundaries
        clause_count = point_count = embedded_count = numbered_count = lettered_count = 0
        current_clause_id = ""

        for provision_index, provision in enumerate(provision_markers):
            provision_type = provision["provision_type"]

            if provision_type == "embedded_article":
                embedded_count += 1
                provision_id = f"{unit_id}_embedded_article_{embedded_count:03}"
                current_clause_id = ""
                boundary_types = {"embedded_article"}
            elif provision_type == "clause":
                clause_count += 1
                provision_id = f"{unit_id}_clause_{clause_count:03}"
                current_clause_id = provision_id
                boundary_types = {"clause", "embedded_article"}
            elif provision_type == "point":
                point_count += 1
                provision_id = f"{unit_id}_point_{point_count:03}"
                boundary_types = {"point", "clause", "embedded_article"}
            elif provision_type == "numbered_item":
                numbered_count += 1
                provision_id = f"{unit_id}_numbered_item_{numbered_count:03}"
                current_clause_id = provision_id
                boundary_types = {"numbered_item", "embedded_article"}
            else:
                lettered_count += 1
                provision_id = f"{unit_id}_lettered_item_{lettered_count:03}"
                boundary_types = {"lettered_item", "numbered_item", "embedded_article"}

            # End at the next equal/higher provision, or earlier at the embedded quotation boundary
            later_provisions = [item for item in provision_markers[provision_index + 1:] if item["provision_type"] in boundary_types]
            provision_start = provision["start_char"]
            provision_end = later_provisions[0]["start_char"] if later_provisions else end_char

            if provision_type == "embedded_article":
                quote_end = find_embedded_quote_end(provision_start, provision_end)
                if quote_end is not None:
                    provision_end = quote_end

            provision_text = text[provision_start:provision_end].strip()


            provisions.append({"provision_id": provision_id, "parent_unit_id": unit_id, "doc_id": path.stem, "provision_type": provision_type, "provision_number": provision["provision_number"], "provision_title": provision["provision_title"], "parent_clause_id": current_clause_id if provision_type in {"point", "lettered_item"} else "", "start_char": provision_start, "end_char": provision_end, "char_count": provision_end - provision_start, "provision_text": provision_text})

    return units, provisions

In [130]:
# Validate the parser on inspected documents before using the complete corpus
parsed_units, parsed_provisions = [], []
inspection_files = sorted(INSPECTION_FOLDER.glob("*.txt"))

for number, path in enumerate(inspection_files, start=1):
    try:
        document_units, document_provisions = parse_legal_document(path)
        parsed_units.extend(document_units)
        parsed_provisions.extend(document_provisions)
        print(f"[{number}/{len(inspection_files)}] {len(document_units):4} units | {len(document_provisions):5} provisions | {path.name}")
    except Exception as error:
        print(f"[{number}/{len(inspection_files)}] ERROR | {path.name} | {error}")

parsed_units_df = pd.DataFrame(parsed_units)
parsed_provisions_df = pd.DataFrame(parsed_provisions)

print("\nDocuments parsed:", len(inspection_files))
print("Legal units created:", len(parsed_units_df))
print("Child provisions created:", len(parsed_provisions_df))

[1/40]    3 units |     0 provisions | 01_2021_TT-BXD_m_488637.txt
[2/40]    4 units |    12 provisions | 02_2025_TT-BTNMT_m_655384.txt
[3/40]    6 units |    41 provisions | 04_2017_QD-TTg_m_346090.txt
[4/40]   28 units |   160 provisions | 04_2024_TT-BTNMT_m_611481.txt
[5/40]   60 units |   450 provisions | 07_2017_QH14_m_355880.txt
[6/40]    5 units |   171 provisions | 07_2025_TT-BTNMT_m_647200.txt
[7/40]    2 units |    45 provisions | 08_2020_QD-TTg_m_444837.txt
[8/40]  165 units |  1902 provisions | 08_2022_ND-CP_m_507203.txt
[9/40]    1 units |   134 provisions | 08_2025_TT-BNNMT_m_666356.txt
[10/40]    4 units |     8 provisions | 09_2025_TT-BYT_m_658051.txt
[11/40]    3 units |     4 provisions | 1009_QD-BCT_m_656831.txt
[12/40]  342 units |  1304 provisions | 101_VBHN-VPQH_m_694902.txt
[13/40]  221 units |   933 provisions | 125_VBHN-VPQH_m_682303.txt
[14/40]    3 units |     0 provisions | 138_NQ-CP_m_544532.txt
[15/40]    2 units |     3 provisions | 144_2024_ND-CP_m_64730

In [86]:
# Check unit coverage, suspiciously small units and repeated Article numbers
unit_summary = pd.crosstab(parsed_units_df["source_filename"], parsed_units_df["unit_type"]).reset_index()
parsed_files = set(parsed_units_df["source_filename"])
documents_without_units = sorted({path.name for path in inspection_files} - parsed_files)
small_units = parsed_units_df[parsed_units_df["char_count"] < 50]
repeated_numbers = parsed_units_df[parsed_units_df.duplicated(["source_filename", "unit_type", "unit_number"], keep=False)].sort_values(["source_filename", "unit_number"])

display(unit_summary)
display(small_units[["source_filename", "unit_type", "unit_number", "unit_title", "char_count"]])
display(repeated_numbers[["source_filename", "unit_type", "unit_number", "unit_title"]])

print("Documents without units:", len(documents_without_units))
print("Units below 50 characters:", len(small_units))
print("Repeated unit-number records:", len(repeated_numbers))
print("Missing documents:", documents_without_units)

unit_type,source_filename,article,roman_section
0,01_2021_TT-BXD_m_488637.txt,3,0
1,02_2025_TT-BTNMT_m_655384.txt,4,0
2,04_2017_QD-TTg_m_346090.txt,6,0
3,04_2024_TT-BTNMT_m_611481.txt,28,0
4,07_2017_QH14_m_355880.txt,60,0
5,07_2025_TT-BTNMT_m_647200.txt,5,0
6,08_2020_QD-TTg_m_444837.txt,2,0
7,08_2022_ND-CP_m_507203.txt,165,0
8,08_2025_TT-BNNMT_m_666356.txt,1,0
9,09_2025_TT-BYT_m_658051.txt,4,0


,source_filename,unit_type,unit_number,unit_title,char_count
324,101_VBHN-VPQH_m_694902.txt,article,44,[4] (abrogated),28
326,101_VBHN-VPQH_m_694902.txt,article,46,[5] (abrogated),28
1498,69_2026_ND-CP_m_699320.txt,article,10,Annulment of clause 6 in Article 23.,49


,source_filename,unit_type,unit_number,unit_title
108,08_2022_ND-CP_m_507203.txt,article,1,Scope
234,08_2022_ND-CP_m_507203.txt,article,1,Entities entitled to incentives and assistance...
275,09_2025_TT-BYT_m_658051.txt,article,2,Effect
277,09_2025_TT-BYT_m_658051.txt,article,2,Methods for determination
654,125_VBHN-VPQH_m_682303.txt,article,32,Part-time employments
842,125_VBHN-VPQH_m_682303.txt,article,32,Labor disputes and labor-related disputes with...
852,1690_QD-TTg_m_114738.txt,roman_section,IV,MAJOR SOLUTIONS
853,1690_QD-TTg_m_114738.txt,roman_section,IV,"MAJOR PROGRAMS, SCHEMES AND PROJECTS"
886,18_VBHN-VPQH_m_699248.txt,article,32,Part-time employments
1074,18_VBHN-VPQH_m_699248.txt,article,32,Labor disputes and labor-related disputes with...


Documents without units: 0
Units below 50 characters: 3
Repeated unit-number records: 12
Missing documents: []


In [131]:
# Review unit metadata and a reproducible sample of complete unit text
display(parsed_units_df[["source_filename", "unit_type", "unit_number", "unit_title", "part_number", "chapter_number", "section_number", "subsection_number", "start_char", "end_char", "char_count"]].head(50))

with pd.option_context("display.max_colwidth", None):
    display(parsed_units_df[["source_filename", "unit_number", "unit_title", "unit_text"]].sample(min(10, len(parsed_units_df)), random_state=42))

,source_filename,unit_type,unit_number,unit_title,part_number,chapter_number,section_number,subsection_number,start_char,end_char,char_count
0,01_2021_TT-BXD_m_488637.txt,article,1,Attached to this Circular are the National Tec...,,,,,932,1058,126
1,01_2021_TT-BXD_m_488637.txt,article,2,"This Circular comes into effect from July 5, 2...",,,,,1058,1269,211
2,01_2021_TT-BXD_m_488637.txt,article,3,"Ministries, ministerial agencies, Governmental...",,,,,1269,1499,230
3,02_2025_TT-BTNMT_m_655384.txt,article,1,The QCVN 01:2025/BTNMT National technical regu...,,,,,2139,2453,314
4,02_2025_TT-BTNMT_m_655384.txt,article,2,Entry into force,,,,,2453,2545,92
5,02_2025_TT-BTNMT_m_655384.txt,article,3,Transitional provisions,,,,,2545,3985,1440
6,02_2025_TT-BTNMT_m_655384.txt,article,4,Implementation,,,,,3985,23844,19859
7,04_2017_QD-TTg_m_346090.txt,article,1,The list of equipment and appliances to which ...,,,,,980,1869,889
8,04_2017_QD-TTg_m_346090.txt,article,2,Roadmap to energy labeling,,,,,1869,3542,1673
9,04_2017_QD-TTg_m_346090.txt,article,3,Roadmap to application of the minimum energy e...,,,,,3542,4620,1078


source_filename unit_number  \
618   101_VBHN-VPQH_m_694902.txt         337   
115   08_2022_ND-CP_m_507203.txt           8   
135   08_2022_ND-CP_m_507203.txt          28   
350   101_VBHN-VPQH_m_694902.txt          70   
1395    37_2026_ND-CP_695602.txt          71   
1447  40_2026_ND-CP_m_695776.txt          10   
669   125_VBHN-VPQH_m_682303.txt          47   
1686   97_2015_QH13_m_299197.txt          14   
1448  40_2026_ND-CP_m_695776.txt          11   
772   125_VBHN-VPQH_m_682303.txt         150   

                                                                                                                     unit_title  \
618                                                                                                           Maritime disputes   
115                                                                         Contents of provincial air quality management plans   
135                                                   Main contents of report on proposal for issuance of environmental license   
350              Providing an account of, investigating, enumerating and reporting maritime occupational accidents and diseases   
1395                             Designation of conformity assessment bodies and accreditation of conformity assessment results   
1447  Minimum competence requirements of organizations and individuals operating culvert headworks and water conveyance systems   
669                                                                                                        Redundancy allowance   
1686                                                                                                                 Collectors   
1448                                         Training and refresher courses in management and operation of hydraulic structures   
772                        Vietnamese employees working overseas, employees of foreign organizations and individuals in Vietnam   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [132]:
# Count units containing each optional hierarchy level
hierarchy_columns = ["part_number", "chapter_number", "section_number", "subsection_number"]
hierarchy_coverage = parsed_units_df[hierarchy_columns].replace("", pd.NA).notna().sum().rename("units_with_value").to_frame()
hierarchy_coverage["percentage"] = (hierarchy_coverage["units_with_value"] / len(parsed_units_df) * 100).round(2)
display(hierarchy_coverage)

,units_with_value,percentage
part_number,0,0.00
chapter_number,1571,92.52
section_number,970,57.13
subsection_number,0,0.00


In [133]:
# Save validation outputs without changing the main corpus
parsed_units_df.drop(columns=["unit_text"]).to_csv(PARSER_REVIEW_FOLDER / "parsed_unit_summary.csv", index=False)
parsed_units_df.to_json(PARSER_REVIEW_FOLDER / "parsed_units.json", orient="records", indent=2, force_ascii=False)
parsed_provisions_df.to_json(PARSER_REVIEW_FOLDER / "parsed_provisions.json", orient="records", indent=2, force_ascii=False)
repeated_numbers.to_csv(PARSER_REVIEW_FOLDER / "repeated_unit_numbers.csv", index=False)
pd.DataFrame({"source_filename": documents_without_units}).to_csv(PARSER_REVIEW_FOLDER / "documents_without_units.csv", index=False)

print("Parser review saved:", PARSER_REVIEW_FOLDER)

Parser review saved: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/article_parser_review


#### Final validation

In [134]:
print("Repeated units:", parsed_units_df["has_repeated_number"].sum())
print("Oversized units:", parsed_units_df["is_oversized"].sum())
print("Embedded Articles:", (parsed_provisions_df["provision_type"] == "embedded_article").sum())
print("Missing provision end offsets:", parsed_provisions_df["end_char"].isna().sum())

Repeated units: 12
Oversized units: 96
Embedded Articles: 76
Missing provision end offsets: 0


In [135]:
# Review repeated Article numbers
repeated_review = parsed_units_df[parsed_units_df["has_repeated_number"]].sort_values(["source_filename", "unit_number", "unit_number_occurrence"])
display(repeated_review[["source_filename", "unit_id", "unit_number", "unit_number_occurrence", "unit_title", "start_char"]])

,source_filename,unit_id,unit_number,unit_number_occurrence,unit_title,start_char
108,08_2022_ND-CP_m_507203.txt,08_2022_ND-CP_m_507203_article_0001,1,1,Scope,895
234,08_2022_ND-CP_m_507203.txt,08_2022_ND-CP_m_507203_article_0127,1,2,Entities entitled to incentives and assistance...,418360
275,09_2025_TT-BYT_m_658051.txt,09_2025_TT-BYT_m_658051_article_0002,2,1,Effect,1518
277,09_2025_TT-BYT_m_658051.txt,09_2025_TT-BYT_m_658051_article_0004,2,2,Methods for determination,43993
654,125_VBHN-VPQH_m_682303.txt,125_VBHN-VPQH_m_682303_article_0032,32,1,Part-time employments,25878
842,125_VBHN-VPQH_m_682303.txt,125_VBHN-VPQH_m_682303_article_0220,32,2,Labor disputes and labor-related disputes with...,185236
852,1690_QD-TTg_m_114738.txt,1690_QD-TTg_m_114738_roman_section_0004,IV,1,MAJOR SOLUTIONS,18374
853,1690_QD-TTg_m_114738.txt,1690_QD-TTg_m_114738_roman_section_0005,IV,2,"MAJOR PROGRAMS, SCHEMES AND PROJECTS",27863
886,18_VBHN-VPQH_m_699248.txt,18_VBHN-VPQH_m_699248_article_0032,32,1,Part-time employments,26142
1074,18_VBHN-VPQH_m_699248.txt,18_VBHN-VPQH_m_699248_article_0220,32,2,Labor disputes and labor-related disputes with...,185323


In [92]:
# Count oversized units and embedded Articles by document
oversized_summary = parsed_units_df[parsed_units_df["is_oversized"]].groupby("source_filename").size().sort_values(ascending=False).rename("oversized_units").to_frame()
embedded_summary = parsed_provisions_df[parsed_provisions_df["provision_type"].eq("embedded_article")].groupby("doc_id").size().sort_values(ascending=False).rename("embedded_articles").to_frame()

display(oversized_summary.head(20))
display(embedded_summary.head(20))

,oversized_units
source_filename,
08_2022_ND-CP_m_507203.txt,31
308_2025_ND-CP_m_688502.txt,12
81_2023_QH15_m_565003.txt,9
37_2026_ND-CP_695602.txt,5
40_2026_ND-CP_m_695776.txt,4
243_2026_ND-CP_m_713627.txt,2
09_2025_TT-BYT_m_658051.txt,2
1690_QD-TTg_m_114738.txt,2
79_2023_ND-CP_m_590378.txt,1


,embedded_articles
doc_id,
308_2025_ND-CP_m_688502,83
243_2026_ND-CP_m_713627,22
37_2026_ND-CP_695602,16
08_2025_TT-BNNMT_m_666356,13
07_2025_TT-BTNMT_m_647200,11
69_2026_ND-CP_m_699320,8
28_2018_QH14_m_388792,6
08_2020_QD-TTg_m_444837,5
203_2025_QH15_m_661982,4


In [136]:
# Confirm complete provision text against the original inspected documents
provision_offset_errors = []

for provision in parsed_provisions:
    path = INSPECTION_FOLDER / f"{provision['doc_id']}.txt"
    if not path.exists():
        provision_offset_errors.append({"provision_id": provision["provision_id"], "error": "source_missing"})
        continue

    source_text = path.read_text(encoding="utf-8", errors="replace")
    start_char, end_char = provision["start_char"], provision["end_char"]

    if not 0 <= start_char < end_char <= len(source_text):
        provision_offset_errors.append({"provision_id": provision["provision_id"], "error": "invalid_offset"})
    elif source_text[start_char:end_char].strip() != provision["provision_text"]:
        provision_offset_errors.append({"provision_id": provision["provision_id"], "error": "text_mismatch"})

provision_offset_errors_df = pd.DataFrame(provision_offset_errors)
print("Provision offset errors:", len(provision_offset_errors_df))
display(provision_offset_errors_df.head(20))

Provision offset errors: 0


""


#### **2.2. Full-corpus legal parsing** 

In [137]:
# Configure input and output folders
MANIFEST_PATH = PROJECT_ROOT / "outputs" / "manifests" / "split_manifest.json"
SPLIT_FOLDER = PROJECT_ROOT / "data" / "splits"
PARSED_FOLDER = PROJECT_ROOT / "data" / "parsed"
REPORT_FOLDER = PROJECT_ROOT / "outputs" / "reports"
PARSED_FOLDER.mkdir(parents=True, exist_ok=True)
REPORT_FOLDER.mkdir(parents=True, exist_ok=True)

# Load one manifest record for each document
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
manifest_records = manifest["documents"] if isinstance(manifest, dict) else manifest
manifest_df = pd.DataFrame(manifest_records)

In [138]:
# Confirm that document IDs and split names are valid
assert manifest_df["doc_id"].is_unique, "Duplicate doc_id values found."
assert set(manifest_df["split"]).issubset({"development", "eval", "test"}), "Unknown split found."

print("Manifest documents:", len(manifest_df))
display(manifest_df["split"].value_counts().rename("document_count").to_frame())

Manifest documents: 403


,document_count
split,
development,283
eval,60
test,60


In [139]:
# Save one JSON record per line
def save_jsonl(records: list, path: Path):
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")

# Load a previously saved JSONL file
def load_jsonl(path: Path) -> list:
    if not path.exists() or path.stat().st_size == 0:
        return []
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

# Select manifest metadata that should be attached to every unit and provision
def get_document_metadata(record: dict) -> dict:
    return {"official_number": record.get("official_number", ""), 
            "cleaned_filename": record["cleaned_filename"], 
            "source_docx_filename": record.get("source_filename", ""), 
            "split": record["split"], 
            "document_type": record.get("document_type", "Unknown"), 
            "esg_domains": record.get("esg_domains", []), 
            "esg_categories": record.get("esg_categories", []), 
            "publication_year": record.get("publication_year"), 
            "regulatory_family_id": record.get("regulatory_family_id", "")}

#### **Final check before running parsing on all documents** 

In [140]:
# Validate the revised rules on the two known difficult documents
CHECK_DOCS = ["09_2026_TT-BNNMT_m_696445", "193_2025_ND-CP_m_673825"]
for doc_id in CHECK_DOCS:
    record = manifest_df[manifest_df["doc_id"].eq(doc_id)].iloc[0]
    path = SPLIT_FOLDER / record["split"] / record["cleaned_filename"]
    test_units, test_provisions = parse_legal_document(path)
    units_df, provisions_df = pd.DataFrame(test_units), pd.DataFrame(test_provisions)
    print(f"\n{doc_id}: {len(units_df)} units | {len(provisions_df)} provisions")
    print("Embedded Articles:", provisions_df["provision_type"].eq("embedded_article").sum())
    display(units_df[["unit_number", "unit_title", "section_number", "section_title", "char_count"]].head(30))


09_2026_TT-BNNMT_m_696445: 28 units | 236 provisions
Embedded Articles: 15


,unit_number,unit_title,section_number,section_title,char_count
0,1,Amendments,,,2851
1,2,Amendments to Article 5,,,1053
2,3,Amendments to Title and some clauses of Articl...,,,2218
3,4,Addition of Article 13a after Article 13,,,8927
4,5,Addition of Article 13b after Article 13a,,,5405
5,6,Addition of Article 13c after Article 13b,,,1864
6,7,Addition of Article 13d after Article 13c,,,2239
7,8,Amendments to Article 16,,,2391
8,9,Addition of Article 18a after Article 18b,,,15820
9,10,Addition of Article 18b after Article 18a,,,14103



193_2025_ND-CP_m_673825: 155 units | 1602 provisions
Embedded Articles: 0


,unit_number,unit_title,section_number,section_title,char_count
0,1,Scope,,,1521
1,2,Regulated entities,,,331
2,3,Definitions,,,1890
3,4,List of mineral groups,,,505
4,5,Regulations on responsibility for contribution...,,,1898
5,6,Geological reconnaissance and mineral survey p...,,,1732
6,7,"Periodic reports on mining operations, reports...",,,690
7,8,"Eligibility, rights, and obligations of organi...",1,PARTICIPATION IN GEOLOGICAL SURVEY FOR MINERALS,1605
8,9,Selection of organizations and individuals for...,1,PARTICIPATION IN GEOLOGICAL SURVEY FOR MINERALS,3188
9,10,Application for geological reconnaissance and ...,2,APPLICATION FOR GEOLOGICAL RECONNAISSANCE AND ...,804


In [141]:
# Confirm the corrected hierarchy and embedded-Article classification
amendment_units, amendment_provisions = parse_legal_document(SPLIT_FOLDER / "development" / "09_2026_TT-BNNMT_m_696445.txt")
mineral_units, _ = parse_legal_document(SPLIT_FOLDER / "development" / "193_2025_ND-CP_m_673825.txt")
amendment_units_df, amendment_provisions_df, mineral_units_df = pd.DataFrame(amendment_units), pd.DataFrame(amendment_provisions), pd.DataFrame(mineral_units)

assert amendment_units_df["unit_number"].tolist() == [str(number) for number in range(1, 29)]
assert amendment_provisions_df["provision_type"].eq("embedded_article").sum() == 15
assert not ((mineral_units_df["unit_number"].eq("3")) & mineral_units_df["unit_title"].eq("SITES OF NATIONAL MINERAL RESERVATION")).any()

section_three = mineral_units_df[mineral_units_df["unit_number"].isin(["18", "19", "20", "21", "22"])]
assert section_three["section_number"].eq("3").all()
assert section_three["section_title"].eq("SITES OF NATIONAL MINERAL RESERVATION").all()

print("Targeted parser validation passed.")
display(section_three[["unit_number", "unit_title", "section_number", "section_title"]])

Targeted parser validation passed.


,unit_number,unit_title,section_number,section_title
17,18,Request and procedures for zoning and declarat...,3,SITES OF NATIONAL MINERAL RESERVATION
18,19,Documents and procedures for amending sites of...,3,SITES OF NATIONAL MINERAL RESERVATION
19,20,Composition of written assessment of impact on...,3,SITES OF NATIONAL MINERAL RESERVATION
20,21,Approval of impact assessment of investment pr...,3,SITES OF NATIONAL MINERAL RESERVATION
21,22,Recovery of minerals under national reservatio...,3,SITES OF NATIONAL MINERAL RESERVATION


In [142]:
# Confirm that Article 4 stops before Chapter II
record = manifest_df[manifest_df["doc_id"].eq("25_2025_TT-BYT_m_669298")].iloc[0]
path = SPLIT_FOLDER / record["split"] / record["cleaned_filename"]
test_units, _ = parse_legal_document(path)
test_article = pd.DataFrame(test_units).query("unit_number == '4'").iloc[0]

print(test_article["unit_text"])
assert "Chapter II" not in test_article["unit_text"]
print("Hierarchy-boundary test passed.")

Article 4. List of diseases requiring long-term treatment
The list of diseases requiring long-term treatment shall be implemented in accordance with Appendix I enclosed with this Circular.
Hierarchy-boundary test passed.


In [146]:
# Confirm that Article 5 is a normal top-level Article
record = manifest_df[manifest_df["doc_id"].eq("20_2023_QH15_m_577937")].iloc[0]
path = SPLIT_FOLDER / record["split"] / record["cleaned_filename"]
test_units, test_provisions = parse_legal_document(path)
test_units_df, test_provisions_df = pd.DataFrame(test_units), pd.DataFrame(test_provisions)

# Confirm that Article 5 exists once as a top-level unit
article_5 = test_units_df[test_units_df["unit_number"].eq("5")]
assert len(article_5) == 1

# Use stable title terms because the source contains the typo "cybersecuriy"
article_5_title = " ".join(article_5.iloc[0]["unit_title"].lower().split())
assert article_5_title.startswith("assurance about")
assert "information security" in article_5_title
assert "e-transactions" in article_5_title

# Confirm that the same Article was not also classified as embedded
embedded_article_5 = test_provisions_df["provision_type"].eq("embedded_article") & test_provisions_df["provision_number"].eq("5")
assert not embedded_article_5.any()

print("Ordinary-Article classification test passed.")
display(article_5[["unit_number", "unit_title", "char_count"]])

Ordinary-Article classification test passed.


,unit_number,unit_title,char_count
4,5,Assurance about cybersecuriy and information s...,481


In [147]:
# Validate genuine embedded Articles and their boundaries in the amendment document
record = manifest_df[manifest_df["doc_id"].eq("09_2026_TT-BNNMT_m_696445")].iloc[0]
path = SPLIT_FOLDER / record["split"] / record["cleaned_filename"]
test_units, test_provisions = parse_legal_document(path)
test_units_df, test_provisions_df = pd.DataFrame(test_units), pd.DataFrame(test_provisions)
embedded_df = test_provisions_df[test_provisions_df["provision_type"].eq("embedded_article")].copy()

# Confirm the expected outer and embedded structures
assert len(test_units_df) == 28
assert len(embedded_df) == 15
assert test_units_df["unit_number"].tolist() == [str(number) for number in range(1, 29)]
assert embedded_df["provision_number"].notna().all()
assert embedded_df["provision_text"].str.len().gt(0).all()

# Confirm that every embedded Article remains inside its parent Article
parent_offsets = test_units_df.set_index("unit_id")[["start_char", "end_char"]]
for _, provision in embedded_df.iterrows():
    parent = parent_offsets.loc[provision["parent_unit_id"]]
    assert parent["start_char"] <= provision["start_char"] < provision["end_char"] <= parent["end_char"]

print("Amendment structure test passed.")
print("Top-level Articles:", len(test_units_df))
print("Embedded Articles:", len(embedded_df))
display(embedded_df[["parent_unit_id", "provision_number", "provision_title", "start_char", "end_char"]])

Amendment structure test passed.
Top-level Articles: 28
Embedded Articles: 15


,parent_unit_id,provision_number,provision_title,start_char,end_char
0,09_2026_TT-BNNMT_m_696445_article_0001,1,Scope,2167,4995
3,09_2026_TT-BNNMT_m_696445_article_0002,5,Groundwater protection,5031,6048
8,09_2026_TT-BNNMT_m_696445_article_0003,13,Organization and operation of council for appr...,6163,6305
13,09_2026_TT-BNNMT_m_696445_article_0004,13a,Organization of appraisal of environmental imp...,8319,17194
45,09_2026_TT-BNNMT_m_696445_article_0005,13b,Organization of appraisal of environmental imp...,17247,22599
61,09_2026_TT-BNNMT_m_696445_article_0006,13c,Organization of appraisal of environmental imp...,22652,24463
68,09_2026_TT-BNNMT_m_696445_article_0007,13d,Approval of result of appraisal of environment...,24516,26702
74,09_2026_TT-BNNMT_m_696445_article_0008,16,Collection of opinions for approval of result ...,26738,29092
78,09_2026_TT-BNNMT_m_696445_article_0009,18a,Main contents of report on proposal for issuan...,29146,39412
112,09_2026_TT-BNNMT_m_696445_article_0010,18b,Applications and procedures for issuing enviro...,44967,59015


In [148]:
# Review the endings of embedded Articles for accidental overflow
with pd.option_context("display.max_colwidth", None):
    display(embedded_df.assign(text_ending=embedded_df["provision_text"].str[-300:])[["provision_number", "provision_title", "text_ending"]])

,provision_number,provision_title,text_ending
0,1,Scope,"rticle 127; clause 1 of Article 145; clause 2 of Article 154; clause 6 of Article 147 and point b, clause 4 of Article 163; point d, clause 14 of Article 168 of Decree No. 08/2022/ND-CP, amended by Decree No. 05/2025/ND-CP dated January 06, 2025 and Decree No. 48/2026/ND-CP dated January 29, 2026.”."
3,5,Groundwater protection,"implementation of measures for management, treatment of wastewater, solid waste and other environmental protection measures for preventing pollutants from dispersing into the groundwater environment according to regulations on management and treatment of wastewater, solid waste and relevant laws.”."
8,13,Organization and operation of council for appraisal of measures for environmental improvement and remediation in mineral mining”.,“Article 13. Organization and operation of council for appraisal of measures for environmental improvement and remediation in mineral mining”.
13,13a,Organization of appraisal of environmental impact assessment report (EIAR),"g any revisions.\n6. The application for appraisal of the EIAR shall be received in person, by post, or electronically through the online public service system;\n7. The application for appraisal of the EIAR includes:\na) An application form for appraisal of the EIAR;\nb) The investment project’s EIAR.”."
45,13b,Organization of appraisal of environmental impact assessment report (EIAR) in the form of establishment of appraisal council,"val without revisions: All appraisal sheets written by council members shall be approved without revisions\nb) No approval: At least 1/3 (one third) of appraisal sheets of council members are not approved;\nc) Approval with revisions: other cases that are not specified in points a, b of this clause.”."
61,13c,Organization of appraisal of environmental impact assessment report (EIAR) in the form of collection of experts’ opinions,"follows\na) Approval without revisions: All respondents’ written comments shall be approved without revisions;\nb) No approval: At least 1/3 (one third) of written comments of respondents are not approved;\nc) Approval with revisions: other cases that are not specified in points a, b of this clause.”."
68,13d,Approval of result of appraisal of environmental impact assessment report (EIAR),"serve as a basis for consideration of approval of the result of appraisal of the project’s EIAR.\n5. The result of processing administrative procedures for approval of the result of appraisal of the EIAR shall be returned in person, by post, or electronically through the online public service system."
74,16,Collection of opinions for approval of result of appraisal of EIAR for investment project discharging wastewater into a hydraulic structure,"rge of wastewater into hydraulic structure; if no written opinion is provided by the prescribed deadline, it shall be deemed as agreement. The written request for provision of opinions and the written opinion shall comply with Form No. 04b and Form No. 04c, Appendix II enclosed with this Circular,”."
78,18a,Main contents of report on proposal for issuance of environmental license,"rovision, industrial cluster, expansion investment project of the operating business or dedicated area for production, business operation and service provision, industrial cluster or operating phased project (hereinafter referred to as “business upon considering issuance of an environmental license”"
112,18b,Applications and procedures for issuing environmental licenses,"3 (two-thirds) of total number of experts approve the application (including attached requirements and conditions, if any), within 05 days from the date of receipt of opinions from experts, the licensing authority or the competent license issuer shall issue the environmental license as prescribed.”."


#### **Parsing**

In [ ]:
# Set True only when the parser or source documents have changed
FORCE_REPARSE = False # set to True when want to rerun parser again 
split_names = ["development", "eval", "test"]

# Check whether complete saved outputs already exist
required_paths = [PARSED_FOLDER / f"{split}_{output}.jsonl" 
                  for split in split_names for output in ["units", "provisions", "parser_log"]]
use_checkpoint = not FORCE_REPARSE and all(path.exists() for path in required_paths)

all_units, all_provisions, processing_log = [], [], []

if use_checkpoint:
    # Reload previous results instead of parsing every document again
    for split_name in split_names:
        all_units.extend(load_jsonl(PARSED_FOLDER / f"{split_name}_units.jsonl"))
        all_provisions.extend(load_jsonl(PARSED_FOLDER / f"{split_name}_provisions.jsonl"))
        processing_log.extend(load_jsonl(PARSED_FOLDER / f"{split_name}_parser_log.jsonl"))
    print("Saved parsing results loaded.")

else:
    # Parse every split separately and save its results
    for split_name in split_names:
        split_units, split_provisions, split_log = [], [], []
        split_records = manifest_df[manifest_df["split"].eq(split_name)].to_dict(orient="records")

        for number, record in enumerate(split_records, start=1):
            path = SPLIT_FOLDER / split_name / record["cleaned_filename"]

            # Log missing source files rather than silently dropping them
            if not path.exists():
                split_log.append({"doc_id": record["doc_id"], "split": split_name, 
                                  "status": "missing_file", "unit_count": 0, 
                                  "provision_count": 0, "error": str(path)})
                print(f"[{number}/{len(split_records)}] MISSING | {path.name}")
                continue

            try:
                document_units, document_provisions = parse_legal_document(path)
                metadata = get_document_metadata(record)

                # Attach document metadata to every parsed record
                document_units = [{**unit, **metadata, "doc_id": record["doc_id"]} for unit in document_units]
                document_provisions = [{**provision, **metadata, "doc_id": record["doc_id"]} for provision in document_provisions]
                split_units.extend(document_units)
                split_provisions.extend(document_provisions)

                status = "ok" if document_units else "no_units"
                split_log.append({"doc_id": record["doc_id"], 
                                  "split": split_name, "status":
                                    status, "unit_count": len(document_units), 
                                    "provision_count": len(document_provisions), 
                                    "error": None})
                print(f"[{number}/{len(split_records)}] {len(document_units):4} units | {len(document_provisions):5} provisions | {path.name}")

            except Exception as error:
                split_log.append({"doc_id": record["doc_id"], 
                                  "split": split_name, "status": "error", 
                                  "unit_count": 0, "provision_count": 0, 
                                  "error": str(error)})
                print(f"[{number}/{len(split_records)}] ERROR | {path.name} | {error}")

        # Save this split so it can be reloaded in future sessions
        save_jsonl(split_units, PARSED_FOLDER / f"{split_name}_units.jsonl")
        save_jsonl(split_provisions, PARSED_FOLDER / f"{split_name}_provisions.jsonl")
        save_jsonl(split_log, PARSED_FOLDER / f"{split_name}_parser_log.jsonl")
        all_units.extend(split_units)
        all_provisions.extend(split_provisions)
        processing_log.extend(split_log)

        print(f"{split_name.upper()} saved: {len(split_units)} units, {len(split_provisions)} provisions")

[1/283]   37 units |   189 provisions | 38_2016_ND-CP_m_321908.txt
[2/283]   23 units |   163 provisions | 35_2016_ND-CP_m_315367.txt
[3/283]   26 units |   160 provisions | 41_2016_ND-CP_m_316497.txt
[4/283]   35 units |   228 provisions | 40_2026_ND-CP_m_695776.txt
[5/283]    5 units |    32 provisions | 34_2018_TT-BNNPTNT_m_524985.txt
[6/283]   21 units |    66 provisions | 33_2014_TT-BNNPTNT_m_266771.txt
[7/283]   10 units |    48 provisions | 31_2015_QD-TTg_m_287591.txt
[8/283]    5 units |    33 provisions | 98_2019_ND-CP_m_666640.txt
[9/283]   29 units |   178 provisions | 84_2019_ND-CP_m_431366.txt
[10/283]   49 units |   329 provisions | 80_2014_ND-CP_m_248127.txt
[11/283]   37 units |   260 provisions | 79_2023_ND-CP_m_590378.txt
[12/283]   20 units |    96 provisions | 94_2025_TT-BNNMT_m_695179.txt
[13/283]   17 units |   165 provisions | 94_2019_ND-CP_m_431980.txt
[14/283]   59 units |   534 provisions | 54_2024_ND-CP_m_614274.txt
[15/283]    3 units |    88 provisions | 48

In [150]:
# Combine the three splits for validation
all_units_df = pd.DataFrame(all_units)
all_provisions_df = pd.DataFrame(all_provisions)
processing_log_df = pd.DataFrame(processing_log)

print("Documents processed:", len(processing_log_df))
print("Legal units:", len(all_units_df))
print("Child provisions:", len(all_provisions_df))
display(processing_log_df["status"].value_counts().rename("document_count").to_frame())

Documents processed: 403
Legal units: 11013
Child provisions: 87693


,document_count
status,
ok,403


In [151]:
# Review metadata for documents where the parser found no legal units

manifest_columns = ["doc_id", "official_number", "document_type", "cleaned_filename", "split"]
manifest_columns = [column for column in manifest_columns if column in manifest_df.columns]

no_unit_documents = processing_log_df[processing_log_df["status"]
                                      .eq("no_units")].merge(manifest_df[manifest_columns], 
                                                             on=["doc_id", "split"], how="left")
display(no_unit_documents)

,doc_id,split,status,unit_count,provision_count,error,official_number,document_type,cleaned_filename


In [152]:
# Check unique IDs, parent relationships and regulatory-family leakage
duplicate_unit_ids = all_units_df["unit_id"].duplicated().sum()
duplicate_provision_ids = all_provisions_df["provision_id"].duplicated().sum()
orphan_provisions = all_provisions_df[~all_provisions_df["parent_unit_id"].isin(set(all_units_df["unit_id"]))]
family_leakage = manifest_df.groupby("regulatory_family_id")["split"].nunique().gt(1).sum()

print("Duplicate unit IDs:", duplicate_unit_ids)
print("Duplicate provision IDs:", duplicate_provision_ids)
print("Provisions without parent units:", len(orphan_provisions))
print("Families crossing splits:", family_leakage)

Duplicate unit IDs: 0
Duplicate provision IDs: 0
Provisions without parent units: 0
Families crossing splits: 0


In [153]:
# Confirm that every unit's offsets reproduce its saved text
# offset = character position of extracted text inside the original cleaned document.
# allow you to determine whether a retrieved chunk overlaps the correct gold Article or supporting provision.
offset_errors = []

for unit in all_units:
    path = SPLIT_FOLDER / unit["split"] / unit["cleaned_filename"]

    if not path.exists():
        offset_errors.append({"unit_id": unit["unit_id"], "error": "source_missing"})
        continue

    source_text = path.read_text(encoding="utf-8", errors="replace")
    start_char, end_char = unit["start_char"], unit["end_char"]

    if not 0 <= start_char < end_char <= len(source_text):
        offset_errors.append({"unit_id": unit["unit_id"], "error": "invalid_offset"})
    elif source_text[start_char:end_char].strip() != unit["unit_text"]:
        offset_errors.append({"unit_id": unit["unit_id"], "error": "text_mismatch"})

offset_errors_df = pd.DataFrame(offset_errors)
print("Offset errors:", len(offset_errors_df))
display(offset_errors_df.head(20))

Offset errors: 0


""


In [154]:
# Treat only parsing failures as problem documents
problem_statuses = ["missing_file", "error", "no_units"]
problem_documents = processing_log_df[processing_log_df["status"].isin(problem_statuses)].copy()

# Identify short units expressing inactive or removed legal provisions
inactive_pattern = r"\b(annul(?:led|ment)?|repeal(?:ed|ment)?|abrogat(?:ed|ion)?)\b"
inactive_mask = (small_units["unit_title"].fillna("") + " " + small_units["unit_text"].fillna("")).str.contains(inactive_pattern, case=False, regex=True)
inactive_small_units = small_units[inactive_mask].copy()
short_units_for_review = small_units[~inactive_mask].copy()

print("Problem documents:", len(problem_documents))
print("Inactive short units retained:", len(inactive_small_units))
print("Other short units for review:", len(short_units_for_review))

display(problem_documents)

# Manually inspect development units only; show counts for eval and test
display(short_units_for_review[short_units_for_review["split"].eq("development")][["doc_id", "unit_type", "unit_number", "unit_title", "char_count"]])
display(inactive_small_units.groupby(["split", "unit_type"]).size().rename("inactive_short_units").to_frame())
display(short_units_for_review.groupby(["split", "unit_type"]).size().rename("short_units_for_review").to_frame())

# Summarize parser outputs without showing eval or test text
display(processing_log_df.groupby(["split", "status"]).size().unstack(fill_value=0))
display(all_units_df.groupby(["split", "unit_type"]).size().unstack(fill_value=0))
display(all_provisions_df.groupby(["split", "provision_type"]).size().unstack(fill_value=0))

Problem documents: 0
Inactive short units retained: 33
Other short units for review: 0


,doc_id,split,status,unit_count,provision_count,error


,doc_id,unit_type,unit_number,unit_title,char_count


,,inactive_short_units
split,unit_type,
development,article,23
test,article,10


,,short_units_for_review
split,unit_type,


status,ok
split,
development,283
eval,60
test,60


unit_type,article,document_fallback,roman_section
split,,,
development,7827,3,17
eval,1380,0,15
test,1771,0,0


provision_type,clause,embedded_article,lettered_item,numbered_item,point
split,,,,,
development,31271,361,12,66,32866
eval,5207,92,0,66,4547
test,6845,116,0,0,6244


In [155]:
# Stop completion if any essential parser validation has failed
assert len(processing_log_df) == len(manifest_df), "Not all manifest documents were processed."
assert processing_log_df["status"].eq("ok").all(), "Some documents were not parsed successfully."
assert duplicate_unit_ids == 0 and duplicate_provision_ids == 0, "Duplicate IDs detected."
assert len(orphan_provisions) == 0, "Provisions without valid parent units detected."
assert family_leakage == 0, "Regulatory-family leakage detected."
assert len(offset_errors_df) == 0, "Text-offset errors detected."
assert len(short_units_for_review) == 0, "Unresolved short units remain."

# Save the essential validation results
report = {"manifest_documents": len(manifest_df), "documents_processed": len(processing_log_df), "processing_status": {status: int(count) for status, count in processing_log_df["status"].value_counts().items()}, "total_units": len(all_units_df), "total_provisions": len(all_provisions_df), "embedded_articles": int(all_provisions_df["provision_type"].eq("embedded_article").sum()), "oversized_units": int(all_units_df["is_oversized"].sum()), "repeated_number_units": int(all_units_df["has_repeated_number"].sum()), "inactive_short_units": len(inactive_small_units), "unresolved_short_units": len(short_units_for_review), "duplicate_unit_ids": int(duplicate_unit_ids), "duplicate_provision_ids": int(duplicate_provision_ids), "orphan_provisions": len(orphan_provisions), "family_leakage": int(family_leakage), "offset_errors": len(offset_errors_df)}

REPORT_PATH = REPORT_FOLDER / "step2_2_parser_report.json"
REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

print("Step 2.2 validation passed.")
print("Parsed datasets:", PARSED_FOLDER)
print("Validation report:", REPORT_PATH)
display(pd.Series(report, name="value").to_frame())

Step 2.2 validation passed.
Parsed datasets: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/data/parsed
Validation report: /Users/tanggiee/Desktop/RAG_AI/esg_rag_project/outputs/reports/step2_2_parser_report.json


,value
manifest_documents,403
documents_processed,403
processing_status,{'ok': 403}
total_units,11013
total_provisions,87693
embedded_articles,569
oversized_units,584
repeated_number_units,66
inactive_short_units,33
unresolved_short_units,0
